# Load Dependencies

In [1]:
# Base libraries
import os
import pandas as pd
import numpy as np
import regex as re
import time
from dotenv import load_dotenv

# classification libraries
from openai import OpenAI, RateLimitError, APIError, Timeout, APIConnectionError, AsyncOpenAI

# retry libraries
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception_type
from collections import deque

# Load Data

In [5]:
# base_file = "../data/COL26/comments_preliminares.csv"
# df_base = pd.read_csv(base_file)
# df_base.info()

# filtered_file = "../data/COL26/comments_class.csv"
# df = pd.read_csv(filtered_file)
# df.info()


file = "../data/ReformaPensional/comments_preliminares.csv"
df = pd.read_csv(file)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22558 entries, 0 to 22557
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   video_id             22558 non-null  object
 1   comment_id           22558 non-null  object
 2   text                 22555 non-null  object
 3   author_name          22558 non-null  object
 4   author_id            22558 non-null  object
 5   published_at         22558 non-null  object
 6   likes                22558 non-null  int64 
 7   is_reply             22558 non-null  bool  
 8   reply_to_comment_id  3901 non-null   object
dtypes: bool(1), int64(1), object(7)
memory usage: 1.4+ MB


In [6]:
class_number = 1

for i in range(1,class_number+1):
    if not f"C{i}" in df.columns:
        df[f"C{i}"] = np.nan
        df[f"S{i}"] = np.nan

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22558 entries, 0 to 22557
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   video_id             22558 non-null  object 
 1   comment_id           22558 non-null  object 
 2   text                 22555 non-null  object 
 3   author_name          22558 non-null  object 
 4   author_id            22558 non-null  object 
 5   published_at         22558 non-null  object 
 6   likes                22558 non-null  int64  
 7   is_reply             22558 non-null  bool   
 8   reply_to_comment_id  3901 non-null   object 
 9   C1                   0 non-null      float64
 10  S1                   0 non-null      float64
dtypes: bool(1), float64(2), int64(1), object(7)
memory usage: 1.7+ MB


# Classification Setup

## Instruction

In [7]:
def comment_prompt(context,subject):  
    return f"""
    Contexto: {context}
    
    Instrucción: Clasifica el siguiente mensaje en la escala de Likert en relación con la siguiente afirmación: \"{subject}\".
    Las opciones son: 1: 'Completamente en contra', 2: 'En contra', 3: 'Neutral', 4: 'A favor', 5: 'Completamente a favor'.
    Tambien quiero que identifiques el sentimiento del mensaje, si es positivo, negativo o neutral. Responde con 3 para positivo, 1 para negativo y 2 para neutral. 

    Solo responde con una de las etiquetas mencionadas sin ningún texto adicional y el sentimiento. Por ejemplo "2,1" para "En contra", sentimiento negativo, o "4,3" (A favor y sentimiento positivo).
    """

def replied_comment_prompt(context,subject, replied_message):
    return f"""
    Contexto: {context}
    
    Instrucción: Clasifica el siguiente mensaje en la escala de Likert en relación con la siguiente afirmación: \"{subject}\".
    Las opciones son: 1: 'Completamente en contra', 2: 'En contra', 3: 'Neutral', 4: 'A favor', 5: 'Completamente a favor'.
    Tambien quiero que identifiques el sentimiento del mensaje, si es positivo, negativo o neutral. Responde con 3 para positivo, 1 para negativo y 2 para neutral. 

    Ten en cuenta tambien que el mensaje al que responde es: \"{replied_message}\"
    Solo responde con una de las etiquetas mencionadas sin ningún texto adicional y el sentimiento. Por ejemplo "2,1" para "En contra", sentimiento negativo, o "4,3" (A favor y sentimiento positivo).
    """

## Context and Subject

In [ ]:
context = "La reforma pensional en Colombia es un tema recurrente en la agenda política y social del país debido a la " \
"necesidad de abordar la sostenibilidad y la equidad del sistema de pensiones. El sistema actual combina un régimen de " \
"prima media (RPM), administrado por el Estado a través de Colpensiones, y un régimen de ahorro individual (RAIS), " \
"manejado por fondos privados. Sin embargo, el acceso a una pensión digna es limitado para muchos trabajadores, " \
"especialmente aquellos en la informalidad o con bajos ingresos. La reforma busca ampliar la cobertura, mejorar la equidad" \
"entre los diferentes regímenes y asegurar la sostenibilidad financiera a largo plazo."

In [10]:
subject = {"C1": "¿Apoya la reforma pensional en Colombia propuesta por el gobierno de Colombia?"}

# subject = {"C1": "¿Cuál es su nivel de apoyo hacia el candidato presidencial Iván Cepeda?",
#            "C2": "¿Cuál es su nivel de apoyo hacia el candidato presidencial Abelardo de la Espriella?",
#            "C3": "¿Cuál es su nivel de apoyo hacia la candidata presidencial Paloma Valencia?",
#            "C4": "¿Cuál es su nivel de apoyo hacia el candidato presidencial Sergio Fajardo?"}

## OpenAI Setup (Async)

In [14]:
import asyncio
import random

load_dotenv()

api_key = os.getenv("OPENAI_DA")

client = AsyncOpenAI(api_key=api_key)

semaphore = asyncio.Semaphore(15)  # ajusta según rate limit

async def async_openAI_classificator(message, prompt, model="gpt-4.1-2025-04-14", retries=3):
    async with semaphore:
        for attempt in range(retries):
            try:
                response = await client.chat.completions.create(
                    model=model,
                    messages=[
                        {
                            "role": "system",
                            "content": prompt
                        },
                        {
                            "role": "user",
                            "content": message
                        }
                    ],
                    max_tokens=10,
                    temperature=0.0,
                    top_p=0.9
                )

                return response.choices[0].message.content.strip(), response.usage.total_tokens

            except Exception as e:
                if attempt == retries - 1:
                    print(f"Failed after {retries} attempts: {e}")
                    return np.nan, 0
                
                await asyncio.sleep(2 ** attempt + random.random())

In [15]:
async def classify_with_id(record, prompt):    
    try: 
        classification, tokens = await async_openAI_classificator(
            record["text"], prompt
        )

        return {
            "id": record["comment_id"],
            "classification": classification,
            "tokens": tokens
        }
    except Exception as e:
        return {
            "id": record["comment_id"],
            "classification": np.nan,
            "tokens": 0
        }

async def process_records(subject, records,type="normal"):
    if type == "normal":
        tasks = [
            classify_with_id(r, comment_prompt(context,subject))
            for r in records
        ]
    elif type == "reply":
        tasks = [
            classify_with_id(r, replied_comment_prompt(context,subject, r['replied_message']))
            for r in records
        ]

    return await asyncio.gather(*tasks)

#results = await process_records(records, basic_prompt(context, subject))

In [17]:
for i in range(1,class_number+1):
    # aux_df = df[df[f"I{i}"] == True]
    subject_i = subject[f"C{i}"]

    # normal_comment = aux_df[aux_df['is_reply'] == False]
    normal_comment = df[(df['is_reply'] == False)]
    normal_records = normal_comment[['comment_id','text']].to_dict(orient='records')

    # print(f"Processing subject {i}: {subject_i} with {len(aux_df)} records")
    print(f"Processing subject {i}: {subject_i} with {len(df)} records")
    normal_results = await process_records(subject_i, normal_records, "normal")

    for item in normal_results:
        id = item['id']
        try:
            classification, sentiment = item['classification'].split(",")
        except Exception as e:
            classification, sentiment = np.nan, np.nan

        df.loc[(df['comment_id'] == id), f'C{i}'] = classification
        df.loc[(df['comment_id'] == id), f'S{i}'] = sentiment


Processing subject 1: ¿Apoya la reforma pensional en Colombia propuesta por el gobierno de Colombia? with 22558 records
Failed after 3 attempts: Out of range float values are not JSON compliant: nan
Failed after 3 attempts: Out of range float values are not JSON compliant: nan
Failed after 3 attempts: Out of range float values are not JSON compliant: nan


C:\Users\mauricio.munoz\AppData\Local\Temp\ipykernel_21044\2781528080.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '5' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[(df['comment_id'] == id), f'C{i}'] = classification
C:\Users\mauricio.munoz\AppData\Local\Temp\ipykernel_21044\2781528080.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[(df['comment_id'] == id), f'S{i}'] = sentiment


In [22]:
for i in range(1,class_number+1):
    # aux_df = df[df[f"I{i}"] == True]
    subject_i = subject[f"C{i}"]

    # reply_comments = aux_df[aux_df['is_reply'] == True]
    reply_comments = df[(df['is_reply'] == True)]
    reply_records = reply_comments[['comment_id','text','reply_to_comment_id']].to_dict(orient='records')

    for record in reply_records:
        # replied_message = df_base.loc[df_base['comment_id'] == record['reply_to_comment_id'], 'text'].values[0]
        replied_message = df.loc[df['comment_id'] == record['reply_to_comment_id'], 'text'].values[0]
        record['replied_message'] = replied_message 

    # print(f"Processing subject {i}: {subject_i} with {len(aux_df)} records")
    print(f"Processing subject {i}: {subject_i} with {len(df)} records")
    results = await process_records(subject_i, reply_records, "reply")

    for item in results:
        id = item['id']
        try:
            classification, sentiment = item['classification'].split(",")
        except Exception as e:
            classification, sentiment = np.nan, np.nan

        df.loc[(df['comment_id'] == id), f'C{i}'] = classification
        df.loc[(df['comment_id'] == id), f'S{i}'] = sentiment


Processing subject 1: ¿Apoya la reforma pensional en Colombia propuesta por el gobierno de Colombia? with 22558 records


In [24]:
df.to_csv("../data/ReformaPensional/comments_class.csv", index=False)